# HW2 Playground

Fill in TODOs as you work through the assignment.
Implement the required sections in `model.py`, and use this notebook to orchestrate and run your solution.

In [54]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from hw2_loader import HW2DataLoader
from model import GradientBoostingModel

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [55]:
# TODO: Load both datasets
loader = HW2DataLoader()

# Heart disease dataset
heart_path = Path('../data/heart.csv')
X_heart, y_heart = loader.get_heart_disease_data(csv_path=heart_path)
print(X_heart.shape, y_heart.value_counts().to_dict())

# Cancer genomics dataset
cancer_path = Path('../data/cancer_genomics.csv')
labels_path = Path('../data/labels_cancer_genomics.csv')
X_cancer, y_cancer = loader.get_cancer_genomics_data(
    csv_path=cancer_path, labels_path=labels_path
)
print(X_cancer.shape, y_cancer.value_counts().to_dict())

def dedup_xy(X: pd.DataFrame, y: pd.Series):
    """
    Remove duplicate (X, y) pairs.
    Keeps the first occurrence.
    """
    df = X.copy()
    df["_label_"] = y.values

    before = len(df)
    df = df.drop_duplicates()
    after = len(df)

    X_dedup = df.drop(columns="_label_")
    y_dedup = df["_label_"]

    return X_dedup, y_dedup, before, after

# ---- Heart disease ----
X_heart, y_heart, n_before, n_after = dedup_xy(X_heart, y_heart)
print(f"Heart disease: {n_before} → {n_after} samples (removed {n_before - n_after})")

# ---- Cancer genomics ----
X_cancer, y_cancer, n_before, n_after = dedup_xy(X_cancer, y_cancer)
print(f"Cancer genomics: {n_before} → {n_after} samples (removed {n_before - n_after})")

# Sanity checks
print("Heart class distribution:", y_heart.value_counts().to_dict())
print("Cancer class distribution:", y_cancer.value_counts().to_dict())

Successfully loaded heart disease data with 1025 rows
(1025, 13) {1: 526, 0: 499}
(801, 5479) {'BRCA': 300, 'KIRC': 146, 'LUAD': 141, 'PRAD': 136, 'COAD': 78}
Heart disease: 1025 → 302 samples (removed 723)
Cancer genomics: 801 → 801 samples (removed 0)
Heart class distribution: {1: 164, 0: 138}
Cancer class distribution: {'BRCA': 300, 'KIRC': 146, 'LUAD': 141, 'PRAD': 136, 'COAD': 78}


In [56]:
# TODO: Initialize your model (adjust params)
model = GradientBoostingModel(
    task="classification",
    max_depth=3,
    learning_rate=0.1,
    n_estimators=200,
    subsample=1.0,
    random_state=42,
    use_scaler=False,   # scaling not necessary for trees, but allowed
)

In [57]:
# TODO: Train/test split + fit (heart)
X_tr, X_te, y_tr, y_te = model.train_test_split(X_heart, y_heart, test_size=0.2, random_state=42)
model.fit(X_tr, y_tr)

In [58]:
# TODO: Evaluate (heart)
# metrics = model.evaluate(...)
# print metrics
metrics = model.evaluate(X_te, y_te)
print("Test metrics:", metrics)

Test metrics: {'accuracy': 0.7213114754098361, 'precision': 0.7666666666666667, 'recall': 0.696969696969697, 'f1': 0.7301587301587302, 'roc_auc': 0.8127705627705628}


In [59]:
# TODO: Cross-validation (heart)
# cv_results = model.cross_validate(...)
# print metrics
cv_results = model.cross_validate(X_heart, y_heart, cv=5)
print("CV metrics:", cv_results)

CV metrics: {'accuracy': {'mean': 0.7813661202185793, 'std': 0.06719698593633945}, 'precision': {'mean': 0.7885324603540923, 'std': 0.07044048004272443}, 'recall': {'mean': 0.8234848484848486, 'std': 0.04362478802693325}, 'f1': {'mean': 0.8050933737111544, 'std': 0.0553252579929604}, 'roc_auc': {'mean': 0.8762468434343434, 'std': 0.04258359891515145}}


In [60]:
# TODO: Feature importance (heart)
feature_importance = model.get_feature_importance(plot=False)
print(feature_importance)


   feature  importance
0       f2    0.327901
1      f11    0.126882
2      f12    0.111340
3       f7    0.082485
4       f9    0.074100
5       f0    0.068439
6       f4    0.049616
7       f8    0.041964
8       f1    0.040327
9       f3    0.036792
10      f6    0.020693
11     f10    0.016694
12      f5    0.002767


In [61]:
# TODO: Hyperparameter tuning (heart)
param_grid = {
    "max_depth": [1, 2, 3],
    "n_estimators": [50, 100, 200, 400],
    "learning_rate": [0.01, 0.05, 0.1, 0.2],
    "subsample": [0.6, 0.8, 1.0],
    "min_samples_leaf": [1, 2, 5],
    "max_features": [None, "sqrt", "log2"],
}

tuning_results = model.tune_hyperparameters(
    X_heart,
    y_heart,
    param_grid,
    cv=3,
    scoring="roc_auc",   # good default for imbalanced-ish binary tasks
)

print(tuning_results["best_params"])
print(tuning_results["best_score"])

{'learning_rate': 0.01, 'max_depth': 3, 'max_features': 'sqrt', 'min_samples_leaf': 5, 'n_estimators': 200, 'subsample': 0.6}
0.9160713414336602


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

X_train_full, X_test, y_train_full, y_test = train_test_split(
    X_cancer, y_cancer,
    test_size=0.2,
    random_state=42,
    stratify=y_cancer
)

cancer_model = GradientBoostingModel(
    task="classification",
    random_state=42,
    use_scaler=False,  # trees don't need scaling
)

tune_out = cancer_model.tune_hyperparameters(
    X=X_train_full,
    y=y_train_full,
    param_grid=param_grid,
    cv=3,
    scoring="f1_macro",
)

print("Best CV score (f1_macro):", tune_out["best_score"])
print("Best params:", tune_out["best_params"])

y_pred = cancer_model.predict(X_test)
print("\nCustom evaluate():", cancer_model.evaluate(X_test, y_test))

print("\nClassification report:")
print(classification_report(y_test, y_pred, digits=4))

labels = np.unique(y_test)
cm = confusion_matrix(y_test, y_pred, labels=labels)

plt.figure(figsize=(8, 6))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    xticklabels=labels,
    yticklabels=labels,
    cbar=False,
)
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Cancer Genomics Confusion Matrix (Test Set) — Tuned GB")
plt.tight_layout()
plt.show()